## import libraries

In [24]:
import pandas as pd
from geotessera import GeoTessera
from pathlib import Path
import os

## read datasets

In [25]:
data_root = Path("_data/crop_country_exports")

In [26]:
def get_inventory(root_path):
    inventory = []

    for file_path in root_path.glob("*.csv"):
        parts = file_path.stem.split('_')
        
        if len(parts) > 3:
            crop = "_".join(parts[:-2])
            country = parts[-2]
            year = int(parts[-1])

        else:  
            crop = parts[0]
            country = parts[-2]
            year = int(parts[-1])

        inventory.append({
            "crop": crop,
            "country": country,
            "year": year,
            "path": file_path
        })

    df = pd.DataFrame(inventory)

    if not df.empty:
        df = df.sort_values(by=["crop", 'year']).reset_index(drop=True)

    return df

In [27]:
available_datasets = get_inventory(data_root)
available_datasets

,crop,country,year,path
0,maize_corn_popcorn,ee,2016,_data/crop_country_exports/maize_corn_popcorn_...
1,maize_corn_popcorn,at,2016,_data/crop_country_exports/maize_corn_popcorn_...
2,maize_corn_popcorn,be,2016,_data/crop_country_exports/maize_corn_popcorn_...
3,maize_corn_popcorn,dk,2016,_data/crop_country_exports/maize_corn_popcorn_...
4,maize_corn_popcorn,bg,2016,_data/crop_country_exports/maize_corn_popcorn_...
5,maize_corn_popcorn,ie,2017,_data/crop_country_exports/maize_corn_popcorn_...
6,maize_corn_popcorn,pt,2017,_data/crop_country_exports/maize_corn_popcorn_...
7,maize_corn_popcorn,pt,2018,_data/crop_country_exports/maize_corn_popcorn_...
8,maize_corn_popcorn,bg,2018,_data/crop_country_exports/maize_corn_popcorn_...
9,maize_corn_popcorn,dk,2018,_data/crop_country_exports/maize_corn_popcorn_...


## filter dataframe

In [28]:
year = 2019
crop = 'potatoes'

In [29]:
def filter_df(df, crop, year):
    return df[(df['crop'] == crop) & (df['year'] == year)]

In [30]:
filtered_dataset = filter_df(available_datasets, crop, year)
filtered_dataset

,crop,country,year,path
35,potatoes,bg,2019,_data/crop_country_exports/potatoes_bg_2019.csv
36,potatoes,pt,2019,_data/crop_country_exports/potatoes_pt_2019.csv
37,potatoes,dk,2019,_data/crop_country_exports/potatoes_dk_2019.csv
38,potatoes,ie,2019,_data/crop_country_exports/potatoes_ie_2019.csv
39,potatoes,at,2019,_data/crop_country_exports/potatoes_at_2019.csv
40,potatoes,be,2019,_data/crop_country_exports/potatoes_be_2019.csv
41,potatoes,ee,2019,_data/crop_country_exports/potatoes_ee_2019.csv


In [31]:
def create_dataset(df):
    country_dfs = {}

    for country, group in df.groupby('country'):
        country_dfs[country] = pd.concat([pd.read_csv(p) for p in group['path']], ignore_index=True)
    
    return country_dfs

In [32]:
dataset = create_dataset(filtered_dataset)
dataset.keys()

dict_keys(['at', 'be', 'bg', 'dk', 'ee', 'ie', 'pt'])

## retrieve embeddings

In [33]:
gt = GeoTessera(embeddings_dir="_data/embeddings_dir")

In [34]:
def process_embeddings(gdf, country_id, year, include_metadate=True):
    coords = list(zip(gdf['long'], gdf['lat']))

    print(f"Extracting embeddings for {country_id}...")
    embeddings, metadata_list = gt.sample_embeddings_at_points(coords, 
                                                               year=year, 
                                                               include_metadata=include_metadate)
    print(f"Embeddings extracted for {country_id}!")
    return embeddings, metadata_list, coords, country_id

In [35]:
def emb_meta_to_df(embedding, metadata, coords, crop_name, country_id, year):
    print(f"Converting information to a dataframe for {country_id}...")
    
    safe_metadata = [m if m is not None else {} for m in metadata]
    df_row = pd.DataFrame(safe_metadata)

    safe_embeddings = []
    for e in embedding:
        if e is not None:
            safe_embeddings.append(e.flatten())
        else:
            safe_embeddings.append(None) 
    
    df_row['embedding'] = safe_embeddings
    df_row['long_lat'] = coords
    df_row['crop'] = crop_name
    df_row['country_id'] = country_id
    df_row['year'] = year

    # df_row = df_row.fillna("-")

    print(f"Information converted for {country_id}! Total rows: {len(df_row)}")
    return df_row

# after 
# oi = embeddings2.flatten()
# oi.reshape(1, 128).shape

In [36]:
# to avoid running everything at the same time and run out of memory
keys = ['dk', 'ee', 'ie', 'pt']
dataset.keys()

dict_keys(['at', 'be', 'bg', 'dk', 'ee', 'ie', 'pt'])

In [37]:
result = []

for key in keys:
# for key in dataset.keys():
    country = dataset[key]['country_id'][0]
    gdf = dataset[key]
    embd, meta, coords, country_id = process_embeddings(gdf, country, year)
    result_df = emb_meta_to_df(embd, meta, coords, crop, country_id, year)

    result.append(result_df)

Extracting embeddings for dk...


1 points fall in tiles without coverage (will be returned as NaN)


Embeddings extracted for dk!
Converting information to a dataframe for dk...
Information converted for dk! Total rows: 500
Extracting embeddings for ee...
Embeddings extracted for ee!
Converting information to a dataframe for ee...
Information converted for ee! Total rows: 500
Extracting embeddings for ie...
Embeddings extracted for ie!
Converting information to a dataframe for ie...
Information converted for ie! Total rows: 500
Extracting embeddings for pt...
Embeddings extracted for pt!
Converting information to a dataframe for pt...
Information converted for pt! Total rows: 500


## export data

In [38]:
export_folder = "_data/embeddings_exports"

if not os.path.exists(export_folder):
    os.makedirs(export_folder)
    print(f"Created directory: {export_folder}")

for df in result:
    country = df['country_id'].iloc[0]
    file_name = f"{crop}_{country}_{year}_embedding.parquet"
    full_path = os.path.join(export_folder, file_name)
    df.to_parquet(full_path, index=False, engine='pyarrow')
    print(f"Saved: {full_path}")

# To load it back later:
# df = pd.read_parquet('country_crop_embeddings.parquet')

Saved: _data/embeddings_exports/potatoes_dk_2019_embedding.parquet
Saved: _data/embeddings_exports/potatoes_ee_2019_embedding.parquet
Saved: _data/embeddings_exports/potatoes_ie_2019_embedding.parquet
Saved: _data/embeddings_exports/potatoes_pt_2019_embedding.parquet
